# Hausse de loyer 2026 - Collection Équinoxe

**Question :** de combien les loyers de Collection Équinoxe augmenteront-ils en 2026?

| | |
|---|---|
| **Définition retenue** | croissance annualisée du **loyer effectif** (`sRentEffective`) **à unité constante** : médiane, sur tous les baux qui débutent en 2026 (renouvellements et relocations, six immeubles), de la hausse de chaque bail par rapport au bail précédent du même appartement |
| **Estimation 2026** | **3,6 %** (fourchette 2,5 % à 4,6 %) |
| **Méthode** | gradient boosting entraîné sur environ 3 000 paires de baux, qui prédit la hausse de chaque bail de 2026 puis en prend la médiane |
| **Validation** | backtest 2023-2025 : erreur absolue moyenne de 1,1 point, contre 1,4 pour la meilleure extrapolation de tendance |

### Plan, aligné sur les critères d'évaluation

| Section | Critère des consignes |
|---|---|
| 1. Définir la hausse | Définition de la hausse |
| 2. Analyser les données | Analyse des données et effet de mix |
| 3. Mesurer à unité constante | Appariement à unité constante |
| 4. Concessions | Concessions |
| 5. Renouvellements, relocations et Ontario | Renouvellements vs relocations (+ bonus Ontario) |
| 6. Données publiques | Données externes |
| 7. Prévoir et valider | Prévision et backtest (+ bonus : par immeuble, par chambres, fourchette, tableau de bord) |
| 8. Hypothèses et limites, 9. Références | |

## 0. Configuration et chargement

Toutes les constantes sont déclarées ici, une seule fois, avec leur justification : aucun nombre n'apparaît « en dur » plus bas.
Les fichiers sources ne sont jamais modifiés; toute transformation se fait en mémoire.

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))   # évite un avertissement de joblib sous Windows
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

sys.path.insert(0, "public_data")
from extract_cmhc import MARKETS as CMHC_MARKET_LABELS, extract_public_data, number, read_workbook

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

# --- Fichiers (lecture seule) ---
PRIVATE_DIR = Path("private_data")
RAW_PUBLIC_DIR = Path("raw_public_data")
LISTINGS_PATH = PRIVATE_DIR / "equinoxe_listings.csv"
LEASES_PATH = PRIVATE_DIR / "equinoxe_lease_history.csv"
CONCESSIONS_PATH = PRIVATE_DIR / "equinoxe_concessions.csv"
ASKING_PATH = PRIVATE_DIR / "equinoxe_asking_history.csv"
REGULATORY_RATES_PATH = Path("public_data") / "regulatory_rates.csv"
CPI_PATH = RAW_PUBLIC_DIR / "IPC.csv"
LABOUR_PATH = RAW_PUBLIC_DIR / "msrche-du-travail.csv"
DEMOGRAPHY_PATH = RAW_PUBLIC_DIR / "evolution-demographique.csv"
MODEL_PATH = Path("models") / "modele_d_gbm.joblib"

# --- Clé d'unité ---
UNIT_KEY = ["sPropCode", "sUnitCode"]       # 1 pour 1 avec hUnit et sLink (section 2a)
WRONG_UNIT_KEY = ["sSite", "sUnitCode"]     # clé interdite, utilisée une seule fois pour montrer l'erreur (3b)

# --- Appariement ---
MIN_GAP_YEARS = 0.5        # paires plus rapprochées : issues d'un bail de 6 mois, non comparables à un bail annuel (3a)
MAX_GAP_YEARS = 2.5        # paires plus espacées : l'unité a pu être rénovée ou rester vide longtemps
GAP_BINS_YEARS = [0, MIN_GAP_YEARS, 0.95, 1.05, 1.6, MAX_GAP_YEARS, np.inf]   # tranches d'écart pour décrire les paires
DAYS_PER_YEAR = 365.25
PCT = 100

# --- Codes et libellés ---
RENEWAL, TURNOVER = 1, 0                  # sRenewal : 1 = locataire en place, 0 = relocation
QUEBEC, ONTARIO = "Quebec", "Ontario"     # valeurs de sState
PROMO_CHARGE_CODE = "PromoPay"
TWO_BEDROOMS = 2

# --- Périodes ---
FIRST_ANALYSIS_YEAR = 2019        # avant 2019, trop peu de paires de renouvellement
LAST_OBSERVED_YEAR = 2025         # les données s'arrêtent au 2025-12-31
FORECAST_YEAR = LAST_OBSERVED_YEAR + 1
REQUIRED_BACKTEST_YEARS = [2023, 2024, 2025]                      # exigées par les consignes
EXTENDED_BACKTEST_YEARS = [2021, 2022] + REQUIRED_BACKTEST_YEARS  # 3 années ne suffisent pas à départager des variantes
FIRST_PUBLIC_YEAR = 2020          # SCHL : le fichier 2021 donne aussi la variation 2020; les autres séries commencent en 2021

# --- Tolérances et paramètres d'analyse ---
RENT_MATCH_TOLERANCE = 1.0        # $ / mois : deux loyers concordent à un arrondi au dollar près
SIX_MONTH_GAP_YEARS = 0.4         # seuil alternatif qui inclut les baux de 6 mois (sensibilité, 3d)
TRIM_SHARE = 0.05                 # moyenne tronquée : retire 5 % des paires de chaque côté (sensibilité, 3d)
ONTARIO_EXEMPTION_DATE = pd.Timestamp("2018-11-15")   # première occupation après cette date : exempté de la ligne directrice

# --- Modèle (section 7) ---
UNIT_FEATURES = ["sBeds", "sSqft", "sFloor", "gap_years", "is_renewal"]
GBM_PARAMS = {
    "loss": "absolute_error",   # erreur absolue : le modèle vise la médiane, comme notre définition
    "max_iter": 200,
    "learning_rate": 0.05,
    "max_depth": 3,             # arbres peu profonds : peu d'années distinctes
    "min_samples_leaf": 40,     # chaque feuille repose sur au moins 40 paires
    "random_state": 0,
}
COMPARABLE_VARIANT_TOLERANCE = 0.1   # points : variantes dont l'erreur par bail est à moins de 0,1 point du modèle retenu

### Chargement

On charge les quatre tables et on convertit les dates une seule fois. `lease_start_year` (année de `sLeaseFrom`) est l'année
à laquelle on rattache un bail : c'est la date où le nouveau loyer **entre en vigueur**, donc celle qui compte pour les revenus.

In [ ]:
listings = pd.read_csv(LISTINGS_PATH)
leases = pd.read_csv(LEASES_PATH, parse_dates=["sLeaseFrom", "sLeaseTo", "sSignDate", "sAvailable"])
concessions = pd.read_csv(CONCESSIONS_PATH, parse_dates=["sDateFrom", "sDateTo"])
asking = pd.read_csv(ASKING_PATH)

leases["lease_start_year"] = leases["sLeaseFrom"].dt.year
asking["ask_month"] = pd.PeriodIndex(asking["sMonth"], freq="M").to_timestamp()

for name, table in [("listings", listings), ("leases", leases), ("concessions", concessions), ("asking", asking)]:
    print(f"{name:<12} {len(table):>6,} lignes  {table.shape[1]:>2} colonnes")
print(f"\nbaux du {leases['sLeaseFrom'].min():%Y-%m-%d} au {leases['sLeaseFrom'].max():%Y-%m-%d}")

## 1. Définir la hausse mesurée

« La hausse de loyer 2026 » n'est pas un seul chiffre. Les candidats (chiffrés en section 3c) :

| Définition candidate | Ce qu'elle mesure |
|---|---|
| Médiane annuelle de tous les baux | le loyer du bail « typique » de l'année : **mesure surtout le mix d'immeubles** (section 2f) |
| Même unité, loyer contractuel (`sRent`) | la hausse du prix inscrit au bail, pour un même appartement |
| **Même unité, loyer effectif (`sRentEffective`)** | **la hausse de l'argent réellement perçu, pour un même appartement** |
| Renouvellements seulement | la hausse consentie aux locataires en place (TAL au Québec, ligne directrice en Ontario) |
| Relocations seulement | la hausse du prix de marché quand une unité change de locataire |

> **Définition retenue : hausse 2026 = croissance annualisée du loyer effectif à unité constante, médiane de tous les baux
> qui débutent en 2026 (renouvellements et relocations confondus), pour les six immeubles.**

**Pourquoi :**

1. **À unité constante.** La médiane annuelle bondit de 10,8 % en 2023 alors qu'un même appartement augmente d'environ 2 % :
   elle mesure l'arrivée de trois immeubles haut de gamme, pas une hausse payée par les locataires (2f, 3c).
2. **Loyer effectif.** La question d'affaires est de **budgéter les revenus**, c'est-à-dire ce qui est perçu après les mois
   gratuits et autres concessions. Depuis 2024, les deux mesures divergent : en 2025, +7,0 % au bail mais +3,6 % perçu (section 4).
   Citer le contractuel surestimerait la hausse des revenus de plus de trois points.
3. **Tous les baux, mais modélisés selon leur type.** Les revenus 2026 viendront des renouvellements et des relocations, donc le
   chiffre les combine. Comme ils obéissent à des règles différentes (section 5), le modèle connaît le type de bail et
   pondère les deux cas.
4. **Médiane, annualisée, rattachée à `sLeaseFrom`.** La médiane résiste aux paires atypiques, l'annualisation rend comparables
   des baux de durées différentes, et `sLeaseFrom` est la date où le nouveau loyer s'applique.

**Indicateurs secondaires** rapportés à côté : renouvellements en loyer contractuel (ce que le TAL et l'Ontario encadrent),
relocations (référence pour les loyers affichés), prévision par immeuble et par nombre de chambres.

## 2. Analyser les données

Les descriptions ne correspondent pas toujours aux données. Chaque sous-section teste une hypothèse et se termine par une
**lecture** et une **conséquence**. On garde les noms Yardi pendant les vérifications, puis on renomme en 2e.

### 2a. La clé d'unité

In [ ]:
unit_key_checks = pd.Series({
    "hUnit distincts par sPropCode + sUnitCode (max)": leases.groupby(UNIT_KEY)["hUnit"].nunique().max(),
    "clés sPropCode + sUnitCode par hUnit (max)": leases.groupby("hUnit")[UNIT_KEY].nunique().max().max(),
    "clés sPropCode + sUnitCode par sLink (max)": leases.groupby("sLink")[UNIT_KEY].nunique().max().max(),
    "unités en collision sur sSite + sUnitCode": listings.duplicated(WRONG_UNIT_KEY, keep=False).sum(),
    "unités de listings": len(listings),
    "baux en double (même unité, même début)": leases.duplicated(UNIT_KEY + ["sLeaseFrom"]).sum(),
})
unit_key_checks.to_frame("valeur")

**Lecture :** `sPropCode + sUnitCode`, `hUnit` et `sLink` sont trois clés équivalentes (1 pour 1). `sSite + sUnitCode`
fusionnerait 248 appartements distincts : un site regroupe plusieurs propriétés dont les numéros d'unité se chevauchent.
Aucun doublon.

**Conséquence :** tout appariement se fait sur `UNIT_KEY = ["sPropCode", "sUnitCode"]`.

### 2b. Ce que représentent les colonnes des baux

In [ ]:
first_unit_leases = leases[leases["sTermSeq"] == 1]
lease_column_checks = pd.Series({
    "sAvailable == sLeaseFrom (% des baux)": (leases["sAvailable"] == leases["sLeaseFrom"]).mean() * PCT,
    "baux avec sTermSeq = 1": len(first_unit_leases),
    "unités distinctes": leases.groupby(UNIT_KEY).ngroups,
    "sTermSeq = 1 : % de relocations": first_unit_leases["sRenewal"].eq(TURNOVER).mean() * PCT,
    "sConcession == (sRentEffective < sRent) (% des baux)":
        (leases["sConcession"] == (leases["sRentEffective"] < leases["sRent"])).mean() * PCT,
}).round(1)
print(lease_column_checks.to_string(), end="\n\n")
print("Étiquette sLeaseTerm x durée réelle sTermMonths :")
pd.crosstab(leases["sLeaseTerm"], leases["sTermMonths"].round())

**Lecture :**

| Colonne | Constat | Ce qu'elle est réellement |
|---|---|---|
| `sAvailable` | égale à `sLeaseFrom` pour 100 % des baux | une copie de la date de début |
| `sTermSeq` | autant de `sTermSeq = 1` que d'unités, tous des relocations | le **rang du bail dans l'historique de l'unité**, pas du locataire |
| `sConcession` | vaut 1 exactement quand `sRentEffective < sRent` | un indicateur **dérivé** des deux loyers |
| `sTermMonths` | surtout 12 mois, parfois 6, 18 ou 24 | une durée calculée à partir des dates, fiable |
| `sLeaseTerm` | « 6 Months » ou « Short Term » contiennent des baux de 12 mois | une étiquette commerciale **non fiable** |

**Conséquences :** le type de bail vient de `sRenewal`, jamais de `sTermSeq`; `sConcession` n'est jamais utilisé à côté du
loyer effectif (même information deux fois); les hausses sont **annualisées** avec les dates réelles; `sAvailable` et
`sLeaseTerm` sont ignorés.

### 2c. Le piège `PromoPay`

La table des concessions décrit `PromoPay` comme un montant **mensuel**. On rattache chaque concession au bail de la même unité
dont la période contient son début, puis on exprime son montant en mois de loyer.

In [ ]:
def attach_concessions_to_leases(concession_df, lease_df):
    """Rattache chaque concession au bail de la même unité dont la période contient sDateFrom."""
    lease_columns = UNIT_KEY + ["sLeaseFrom", "sLeaseTo", "sRent"]
    candidates = (concession_df.reset_index(names="concession_id")
                  .merge(lease_df[lease_columns].reset_index(names="lease_id"), on=UNIT_KEY))
    starts_within_lease = candidates["sDateFrom"].between(candidates["sLeaseFrom"], candidates["sLeaseTo"])
    return candidates[starts_within_lease]


matched_concessions = attach_concessions_to_leases(concessions, leases)
concessions_by_code = (concessions.groupby("sChargeCode")
                       .agg(concessions=("sAmount", "size"),
                            median_amount=("sAmount", "median"),
                            median_months=("sMonths", "median"),
                            share_single_month=("sMonths", lambda months: (months == 1).mean()))
                       .sort_values("concessions", ascending=False))
print(f"concessions rattachées à un bail : {len(matched_concessions):,} / {len(concessions):,}\n")
print(concessions_by_code.round(2), end="\n\n")

promo = matched_concessions[matched_concessions["sChargeCode"] == PROMO_CHARGE_CODE]
promo_in_months_of_rent = -promo["sAmount"] / promo["sRent"]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(promo_in_months_of_rent, bins=40)
ax.set_xlabel("montant PromoPay / loyer mensuel du bail (en mois de loyer)")
ax.set_ylabel("concessions")
ax.set_title("PromoPay : un montant unique proche d'un mois de loyer, pas un rabais mensuel")
plt.tight_layout()
plt.show()

**Lecture :** `PromoPay` est inscrit sur **un seul mois** dans 88 % des cas, pour environ **un mois de loyer**. S'il était mensuel,
les résidents habiteraient presque gratuitement : la description est fausse. C'est un **mois gratuit unique à la signature**.
Les autres codes valent environ 130 à 150 $ par mois sur 3 à 12 mois; `freepark`, `freelock` et `freeappl` portent sur des
**accessoires**, pas sur le loyer.

**Conséquence :** `PromoPay` est renommé `one_time_signing_discount`; son effet doit être **étalé sur la durée du bail**
(un mois gratuit sur 12 mois ≈ 8 % de rabais). On vérifie que `sRentEffective` fait exactement cela.

### 2d. Validation de `sRentEffective`

Hypothèse : loyer effectif = loyer contractuel + (somme des concessions du bail, `sAmount × sMonths`) / durée du bail.

In [ ]:
concession_totals = (matched_concessions
                     .assign(total_amount=matched_concessions["sAmount"] * matched_concessions["sMonths"])
                     .groupby("lease_id")
                     .agg(concession_total=("total_amount", "sum"), concession_count=("total_amount", "size")))
reconstruction = leases.join(concession_totals).fillna({"concession_total": 0, "concession_count": 0})
observed_gap = reconstruction["sRentEffective"] - reconstruction["sRent"]
predicted_gap = reconstruction["concession_total"] / reconstruction["sTermMonths"]
reconstruction["is_reconstructed"] = (observed_gap - predicted_gap).abs() < RENT_MATCH_TOLERANCE

print(f"sRentEffective reconstruit au dollar près : {reconstruction['is_reconstructed'].mean():.1%} des baux\n")
failures = reconstruction[~reconstruction["is_reconstructed"]]
pd.crosstab(observed_gap[failures.index] < 0, failures["concession_count"] > 0,
            rownames=["rabais observé"], colnames=["concessions dans la table"])

**Lecture :** la formule reconstruit `sRentEffective` **au dollar près pour environ 62 % des baux** : une concordance exacte sur
des milliers de baux n'est pas une coïncidence. `sRentEffective` est bien le loyer contractuel diminué des concessions étalées
sur le bail, `PromoPay` compris. Les échecs viennent de la **table des concessions** : rabais sans concession dans l'extrait
(table incomplète), concessions sans rabais (surtout des accessoires non déduits du loyer), rattachement imparfait.

**Conséquence :** on utilise `sRentEffective` tel que calculé par Yardi, plutôt que de le recalculer à partir d'une table incomplète.

### 2e. Renommage des colonnes

**Cette étape modifie les tables** : à partir d'ici, les colonnes portent le nom de ce qu'elles contiennent réellement (2b à 2d).
Les clés et les colonnes descriptives (`sPropCode`, `sUnitCode`, `sBuilding`, `sState`, `sBeds`...) gardent leur nom Yardi, déjà exact.

In [ ]:
LEASE_COLUMN_RENAMES = {
    "sRent": "contract_rent",
    "sRentEffective": "effective_rent",            # contractuel - concessions étalées sur le bail (2d)
    "sConcession": "has_concession",               # = effective_rent < contract_rent (2b)
    "sRenewal": "is_renewal",                      # 1 = locataire en place, 0 = relocation
    "sTermSeq": "unit_lease_rank",                 # rang du bail dans l'historique de l'unité (2b)
    "sTermMonths": "lease_term_months",
    "sLeaseFrom": "lease_start",
    "sLeaseTo": "lease_end",
    "sSignDate": "sign_date",
    "sAvailable": "lease_start_duplicate",         # copie exacte de sLeaseFrom (2b)
    "sLeaseTerm": "lease_term_label_unreliable",   # étiquette incohérente avec la durée réelle (2b)
}
LISTING_COLUMN_RENAMES = {"sRent": "proposed_next_rent"}     # loyer proposé au prochain bail (section 7g)
CONCESSION_COLUMN_RENAMES = {"sAmount": "amount_per_month", "sMonths": "months_applied",
                             "sDateFrom": "concession_start", "sDateTo": "concession_end"}
CHARGE_CODE_LABELS = {
    "PromoPay": "one_time_signing_discount",       # PAS mensuel : environ un mois de loyer, une fois (2c)
    "freerent": "free_rent_monthly",
    "freepark": "free_parking_accessory",
    "freelock": "free_locker_accessory",
    "freeappl": "free_appliances_accessory",
    "Freeothe": "other_uncategorized",
    "Indem": "tenant_indemnity",
    "Referenc": "referral_credit",
}

leases = leases.rename(columns=LEASE_COLUMN_RENAMES)
listings = listings.rename(columns=LISTING_COLUMN_RENAMES)
concessions = concessions.rename(columns=CONCESSION_COLUMN_RENAMES)
concessions["charge_type"] = concessions["sChargeCode"].map(CHARGE_CODE_LABELS)
leases[list(LEASE_COLUMN_RENAMES.values())].head(3)

### 2f. L'effet de composition (mix)

Pourquoi la médiane annuelle de tous les baux ne mesure pas la hausse : on regarde **qui** est loué chaque année.

In [ ]:
analysis_window = leases[leases["lease_start_year"].between(FIRST_ANALYSIS_YEAR, LAST_OBSERVED_YEAR)].copy()
analysis_window["rent_per_sqft"] = analysis_window["contract_rent"] / analysis_window["sSqft"]

mix_by_year = analysis_window.groupby("lease_start_year").agg(
    leases=("contract_rent", "size"),
    buildings=("sBuilding", "nunique"),
    median_rent=("contract_rent", "median"),
    median_sqft=("sSqft", "median"),
    pct_two_bed_plus=("sBeds", lambda beds: (beds >= TWO_BEDROOMS).mean() * PCT),
    median_rent_per_sqft=("rent_per_sqft", "median"))
mix_by_year["naive_median_growth_pct"] = mix_by_year["median_rent"].pct_change() * PCT
print(mix_by_year.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
share_by_building = pd.crosstab(analysis_window["lease_start_year"], analysis_window["sBuilding"], normalize="index") * PCT
share_by_building.plot(kind="bar", stacked=True, ax=axes[0], width=0.85)
axes[0].set(xlabel="année de début du bail", ylabel="% des baux de l'année",
            title="Trois immeubles haut de gamme entrent en 2023-2024")
axes[0].legend(fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))
analysis_window.groupby("sBuilding")["rent_per_sqft"].median().sort_values().plot(kind="barh", ax=axes[1])
axes[1].set(xlabel="loyer contractuel médian au pi² ($)", ylabel="",
            title="Les nouveaux immeubles sont beaucoup plus chers au pi²")
plt.tight_layout()
plt.show()

**Lecture :** jusqu'en 2022, seuls les trois immeubles de Laval génèrent des baux. Le Carlyle et The Met entrent en 2023,
Westpark en 2024. Les unités louées deviennent **plus petites** et comptent moins de 2 chambres et plus, ce qui devrait faire
*baisser* la médiane, et pourtant elle bondit de **10,8 % en 2023** : les nouveaux immeubles sont beaucoup plus chers au pi².
La médiane mélange des produits différents.

**Conséquence :** la médiane annuelle mesure la **composition du portefeuille**, pas la hausse. On compare chaque unité à
elle-même (section 3). **Limite :** avant 2023, l'historique ne couvre que Laval; le backtest 2023 prédit donc Mont-Royal et
Ottawa avec un historique lavallois.

## 3. Mesurer la croissance à unité constante

Trois étapes : **apparier** chaque bail au bail précédent de la même unité (`UNIT_KEY`), **filtrer** les paires non
comparables, **annualiser** :

$$\text{croissance annualisée} = \left(\frac{\text{loyer}}{\text{loyer précédent}}\right)^{1/\text{écart en années}} - 1$$

La puissance (plutôt qu'une division) tient compte de la composition des hausses : 1 000 $ → 1 060 $ en 18 mois donne
3,96 %/an, et non 6 %. Pour chaque paire, on calcule aussi le **rabais du bail précédent** et la **hausse du rabais**
(section 4), et on range la paire dans un **segment** : renouvellement au Québec, renouvellement en Ontario, ou relocation (section 5).

### 3a. Apparier, filtrer, annualiser

In [ ]:
RENT_COLUMNS = ["contract_rent", "effective_rent"]
PREVIOUS_LEASE_COLUMNS = RENT_COLUMNS + ["lease_start"]


def pair_consecutive_leases(lease_df, unit_key=UNIT_KEY):
    """Associe chaque bail au bail précédent de la même unité (colonnes `prev_`); le premier bail d'une unité n'a pas de paire."""
    ordered = lease_df.sort_values(unit_key + ["lease_start"]).copy()
    previous = ordered.groupby(unit_key)[PREVIOUS_LEASE_COLUMNS].shift(1).add_prefix("prev_")
    paired = pd.concat([ordered, previous], axis=1).dropna(subset=["prev_lease_start"])
    paired["gap_years"] = (paired["lease_start"] - paired["prev_lease_start"]).dt.days / DAYS_PER_YEAR
    return paired


def is_comparable(paired):
    """Écart entre MIN_GAP_YEARS et MAX_GAP_YEARS, et quatre loyers positifs."""
    has_comparable_gap = paired["gap_years"].between(MIN_GAP_YEARS, MAX_GAP_YEARS, inclusive="neither")
    rent_columns = RENT_COLUMNS + [f"prev_{rent}" for rent in RENT_COLUMNS]
    return has_comparable_gap & paired[rent_columns].gt(0).all(axis=1)


def annualized_growth_pct(paired, rent_column):
    ratio = paired[rent_column] / paired[f"prev_{rent_column}"]
    return (ratio ** (1 / paired["gap_years"]) - 1) * PCT


def discount_pct(effective_rent, contract_rent):
    """Part du loyer contractuel qui n'est pas perçue (%)."""
    return (1 - effective_rent / contract_rent) * PCT


def build_comparable_pairs(lease_df):
    """Paires comparables avec croissances annualisées, rabais du bail précédent, hausse du rabais et segment."""
    paired = pair_consecutive_leases(lease_df)
    pairs = paired[is_comparable(paired)].copy()
    for rent in RENT_COLUMNS:
        pairs[f"{rent}_growth_pct"] = annualized_growth_pct(pairs, rent)
    pairs["prev_discount_pct"] = discount_pct(pairs["prev_effective_rent"], pairs["prev_contract_rent"])
    pairs["discount_change_pts"] = discount_pct(pairs["effective_rent"], pairs["contract_rent"]) - pairs["prev_discount_pct"]
    is_renewal, is_ontario = pairs["is_renewal"].eq(RENEWAL), pairs["sState"].eq(ONTARIO)
    pairs["segment"] = np.select([is_renewal & ~is_ontario, is_renewal & is_ontario],
                                 ["renewal_QC", "renewal_ON"], default="turnover")
    return pairs


all_pairs = pair_consecutive_leases(leases)
comparable_pairs = build_comparable_pairs(leases)
pairs = comparable_pairs[comparable_pairs["lease_start_year"] >= FIRST_ANALYSIS_YEAR]

all_pairs["gap_bucket"] = pd.cut(all_pairs["gap_years"], bins=GAP_BINS_YEARS)
all_pairs["raw_contract_growth_pct"] = (all_pairs["contract_rent"] / all_pairs["prev_contract_rent"] - 1) * PCT
print(pd.Series({
    "baux": len(leases),
    "paires (bail précédent trouvé)": len(all_pairs),
    "paires comparables (filtres)": len(comparable_pairs),
    f"paires comparables depuis {FIRST_ANALYSIS_YEAR}": len(pairs),
}).to_string(), end="\n\n")
all_pairs.groupby("gap_bucket", observed=True).agg(
    pairs=("gap_years", "size"), median_raw_contract_growth_pct=("raw_contract_growth_pct", "median")).round(2)

**Lecture :** la grande majorité des paires sont espacées d'**un an**, le cas d'un bail de 12 mois suivi du suivant.
La hausse **brute** est à peu près la même quel que soit l'écart (environ 4,5 %) : le loyer augmente **par bail**, pas au
prorata du temps. Annualiser ramène donc les paires de 18-24 mois à leur vrai rythme annuel, ce qui compte pour un budget annuel.
Le filtre retire les 62 paires de moins de 6 mois (baux de 6 mois, dont la hausse annualisée doublerait artificiellement);
aucune paire ne dépasse 2,5 ans, mais le seuil protège la méthode.

### 3b. Ce que ferait la mauvaise clé (`sSite + sUnitCode`)

On apparie **une seule fois** sur la clé interdite, pour montrer l'effet.

In [ ]:
wrong_key_pairs = pair_consecutive_leases(leases, WRONG_UNIT_KEY)
raw_growth = lambda paired: (paired["contract_rent"] / paired["prev_contract_rent"] - 1) * PCT
pd.DataFrame({
    "sPropCode + sUnitCode": raw_growth(all_pairs).describe(),
    "sSite + sUnitCode (interdit)": raw_growth(wrong_key_pairs).describe(),
}).round(2)

**Lecture :** la mauvaise clé crée des paires qui n'existent pas et compare un appartement à un **autre** appartement du même
site : on voit apparaître des « baisses » et des « hausses » absurdes, et un écart-type plusieurs fois plus grand. Avec la bonne
clé, toutes les variations restent dans une plage plausible.

### 3c. Résultats : les définitions candidates chiffrées

In [ ]:
def median_by_year(pair_df, column, mask=None):
    selected = pair_df if mask is None else pair_df[mask]
    return selected.groupby("lease_start_year")[column].median()


analysis_years = list(range(FIRST_ANALYSIS_YEAR, LAST_OBSERVED_YEAR + 1))
candidate_definitions = pd.DataFrame({
    "naive_median_all_leases": mix_by_year["naive_median_growth_pct"],
    "same_unit_contract": median_by_year(pairs, "contract_rent_growth_pct"),
    "same_unit_effective": median_by_year(pairs, "effective_rent_growth_pct"),
    "renewal_effective": median_by_year(pairs, "effective_rent_growth_pct", pairs["is_renewal"].eq(RENEWAL)),
    "turnover_effective": median_by_year(pairs, "effective_rent_growth_pct", pairs["is_renewal"].eq(TURNOVER)),
    "pairs": pairs.groupby("lease_start_year").size(),
}).loc[analysis_years]
yearly_target = candidate_definitions["same_unit_effective"]   # la cible (section 1)
print(candidate_definitions.round(2))

fig, ax = plt.subplots(figsize=(10, 4.5))
for column, style, label in [("naive_median_all_leases", "o:", "médiane naïve de tous les baux"),
                             ("same_unit_contract", "o-", "même unité, loyer contractuel"),
                             ("same_unit_effective", "s-", "même unité, loyer effectif (définition retenue)")]:
    ax.plot(analysis_years, candidate_definitions[column], style, lw=2, label=label)
ax.axhline(0, color="grey", lw=0.8)
ax.set(xlabel="année de début du nouveau bail", ylabel="croissance annuelle (%)",
       title="La médiane naïve mesure le mix; contractuel et effectif divergent depuis 2024")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Lecture :** la hausse réelle d'un même appartement est bien plus modeste que la médiane naïve (2023 : +10,8 % naïf contre
environ 2 % à unité constante). Contractuel et effectif évoluent ensemble jusqu'en 2023, puis divergent (section 4).
La cible oscille entre 1,6 % et 4,6 % sur 2019-2025, avec un creux en 2023-2024.

### 3d. Sensibilité aux choix de méthode

On recalcule la croissance effective en changeant **un choix à la fois**.

In [ ]:
all_pairs["raw_effective_growth_pct"] = (all_pairs["effective_rent"] / all_pairs["prev_effective_rent"] - 1) * PCT
all_pairs["annualized_effective_growth_pct"] = annualized_growth_pct(all_pairs, "effective_rent")
within_max_gap = all_pairs["gap_years"] < MAX_GAP_YEARS
reference_pairs = all_pairs[within_max_gap & (all_pairs["gap_years"] > MIN_GAP_YEARS)]
with_six_month_pairs = all_pairs[within_max_gap & (all_pairs["gap_years"] > SIX_MONTH_GAP_YEARS)]


def trimmed_mean(values):
    low, high = values.quantile([TRIM_SHARE, 1 - TRIM_SHARE])
    return values[values.between(low, high)].mean()


by_year = lambda pair_df: pair_df.groupby("lease_start_year")
method_sensitivity = pd.DataFrame({
    "référence (médiane, annualisée, écart > 0,5 an)": by_year(reference_pairs)["annualized_effective_growth_pct"].median(),
    "inclure les baux de 6 mois": by_year(with_six_month_pairs)["annualized_effective_growth_pct"].median(),
    "sans annualisation": by_year(reference_pairs)["raw_effective_growth_pct"].median(),
    "moyenne au lieu de médiane": by_year(reference_pairs)["annualized_effective_growth_pct"].mean(),
    "moyenne tronquée": by_year(reference_pairs)["annualized_effective_growth_pct"].apply(trimmed_mean),
}).loc[REQUIRED_BACKTEST_YEARS].T
method_sensitivity["écart max vs référence"] = method_sensitivity.sub(method_sensitivity.iloc[0]).abs().max(axis=1)
method_sensitivity.round(2)

**Lecture :** aucune variante ne s'écarte de plus d'environ un quart de point de la référence : **la mesure est robuste** aux
choix de filtre, d'annualisation et de statistique. La moyenne simple est un peu plus haute, tirée par quelques fortes hausses,
ce qui justifie la médiane.

## 4. Concessions : loyer contractuel ou loyer effectif?

Pour chaque paire, le mécanisme est exact :

$$\frac{\text{effectif}_t}{\text{effectif}_{t-1}} = \frac{\text{contractuel}_t}{\text{contractuel}_{t-1}} \times \frac{1 - \text{rabais}_t}{1 - \text{rabais}_{t-1}}$$

Un rabais **stable** ne crée aucun écart entre les deux croissances; seul un rabais **qui grossit** d'un bail à l'autre en crée un.
On mesure donc la fréquence des concessions, leur taille, et surtout leur **variation sur les mêmes appartements**.

In [ ]:
leases_in_window = leases[leases["lease_start_year"].between(FIRST_ANALYSIS_YEAR, LAST_OBSERVED_YEAR)]
leases_in_window = leases_in_window.assign(
    discount_pct=discount_pct(leases_in_window["effective_rent"], leases_in_window["contract_rent"]))
concessions_by_year = pd.DataFrame({
    "pct_leases_with_concession": leases_in_window.groupby("lease_start_year")["has_concession"].mean() * PCT,
    "mean_discount_pct": leases_in_window.groupby("lease_start_year")["discount_pct"].mean(),
    "contract_growth": candidate_definitions["same_unit_contract"],
    "effective_growth": candidate_definitions["same_unit_effective"],
    "median_discount_change_pts": median_by_year(pairs, "discount_change_pts"),
    "pct_pairs_discount_up": pairs.groupby("lease_start_year")["discount_change_pts"].apply(lambda change: (change > 0).mean() * PCT),
})
concessions_by_year["growth_gap_pts"] = concessions_by_year["contract_growth"] - concessions_by_year["effective_growth"]
print(concessions_by_year.round(2), end="\n\n")

discount_change_by_type = (pairs[pairs["lease_start_year"] >= REQUIRED_BACKTEST_YEARS[0]]
                           .pivot_table(index="lease_start_year", columns="is_renewal", values="discount_change_pts", aggfunc="median")
                           .rename(columns={RENEWAL: "renouvellement", TURNOVER: "relocation"}))
print("Hausse médiane du rabais (points), par type de bail :")
print(discount_change_by_type.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
concessions_by_year[["contract_growth", "effective_growth"]].plot(ax=axes[0], marker="o")
axes[0].legend(["loyer contractuel", "loyer effectif"])
axes[0].set(xlabel="année du nouveau bail", ylabel="croissance médiane (%)", title="Mêmes appartements, deux loyers")
concessions_by_year[["growth_gap_pts", "median_discount_change_pts"]].plot(ax=axes[1], marker="o")
axes[1].legend(["écart contractuel - effectif (points)", "hausse médiane du rabais (points)"])
axes[1].set(xlabel="année du nouveau bail", ylabel="points", title="L'écart suit la hausse du rabais d'un bail à l'autre")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Lecture : pourquoi les deux croissances divergent à partir de 2024.**

1. **Les concessions deviennent à la fois plus fréquentes et plus généreuses** : la part des baux avec concession passe de 44-57 %
   (2019-2023) à 73 % en 2024 puis 85 % en 2025, et le rabais moyen de 2-4 % à 9,1 % du loyer contractuel.
2. **Sur les mêmes appartements, le rabais grossit d'un bail à l'autre** : hausse médiane nulle en 2023, puis +1,2 point en 2024
   et +3,0 points en 2025 (le rabais augmente pour 86 % des paires de 2025). C'est presque exactement l'écart entre les deux
   croissances : 1,6 point en 2024 et 3,5 points en 2025 (courbe de droite).
3. **Renouvellements et relocations sont touchés de la même façon** : les concessions servent autant à **retenir** les locataires
   en place qu'à en attirer de nouveaux. Elles se comportent comme une **politique commerciale globale**, présente dans tous les
   immeubles, anciens compris.

En résumé : depuis 2024, Équinoxe continue d'augmenter les loyers inscrits aux baux (+7,0 % en 2025), mais en redonne une part
croissante sous forme de concessions; le loyer perçu ne monte que de 3,6 %.

**Conséquences :** (1) on prévoit le **loyer effectif** (section 1); (2) la **taille du rabais au bail précédent** devient une
variable clé du modèle : un appartement loué avec un gros rabais peut voir ce rabais reculer au bail suivant, et inversement (section 7).

## 5. Renouvellements, relocations et règles provinciales

Le type de bail est celui du **nouveau** bail de chaque paire (`is_renewal`). Les règles diffèrent :

| Cas | Québec (5 immeubles) | Ontario (The Met) |
|---|---|---|
| Renouvellement | hausse encadrée par le **TAL** : taux indicatif annuel, qui ne s'impose que si le locataire conteste | **ligne directrice** provinciale, sauf pour les logements occupés pour la première fois après le 15 novembre 2018, **exemptés** du plafond |
| Relocation | loyer de marché (avec la clause G du bail) | loyer de marché, sans plafond |

**On n'applique jamais le plafond d'une province à l'autre**, et on n'écrête aucune hausse : on mesure ce qui a été signé.

In [ ]:
SEGMENT_LABELS = {"renewal_QC": "renouvellement Québec", "renewal_ON": "renouvellement Ontario", "turnover": "relocation"}
recent_pairs = pairs[pairs["lease_start_year"] >= EXTENDED_BACKTEST_YEARS[0]]
by_segment = recent_pairs.pivot_table(index="lease_start_year", columns="segment", aggfunc="median",
                                      values=["contract_rent_growth_pct", "effective_rent_growth_pct"])
segment_counts = recent_pairs.pivot_table(index="lease_start_year", columns="segment", values="effective_rent_growth_pct", aggfunc="size")
print("Paires par segment :")
print(segment_counts.fillna(0).astype(int), end="\n\n")
print("Croissance médiane (%) :")
print(by_segment.round(2))

### 5a. Renouvellements québécois et taux du TAL

Si les renouvellements suivaient le TAL, la « prime » (hausse contractuelle observée - taux du TAL) serait stable.

In [ ]:
regulatory_rates = pd.read_csv(REGULATORY_RATES_PATH).set_index("year")
renewal_vs_regulation = pd.DataFrame({
    "renewal_QC_contract_growth": by_segment[("contract_rent_growth_pct", "renewal_QC")],
    "tal_rate": regulatory_rates["tal_unheated_pct"],
    "renewal_ON_contract_growth": by_segment[("contract_rent_growth_pct", "renewal_ON")],
    "ontario_guideline": regulatory_rates["ontario_guideline_pct"],
}).loc[EXTENDED_BACKTEST_YEARS]
renewal_vs_regulation["premium_over_tal_pts"] = (renewal_vs_regulation["renewal_QC_contract_growth"]
                                                 - renewal_vs_regulation["tal_rate"])
renewal_vs_regulation["premium_over_guideline_pts"] = (renewal_vs_regulation["renewal_ON_contract_growth"]
                                                       - renewal_vs_regulation["ontario_guideline"])
renewal_vs_regulation.round(2)

**Lecture (Québec) :** la prime est **instable** : environ +4 points en 2021-2022 (TAL très bas, Équinoxe augmente bien au-dessus),
puis proche de 0 depuis 2023. Depuis que le TAL est élevé, les renouvellements québécois le suivent à peu près; avant, ils l'ignoraient.
Le taux du TAL n'est qu'indicatif : on ne l'impose donc pas comme plafond, il entre dans le modèle comme **une variable**.

### 5b. The Met : la ligne directrice ontarienne s'applique-t-elle?

In [ ]:
the_met_leases = leases[leases["sState"] == ONTARIO]
the_met_renewals = pairs[pairs["segment"] == "renewal_ON"]
pd.Series({
    "premier bail observé à The Met": f"{the_met_leases['lease_start'].min():%Y-%m-%d}",
    "date d'exemption ontarienne (première occupation après)": f"{ONTARIO_EXEMPTION_DATE:%Y-%m-%d}",
    "renouvellements appariés à The Met": len(the_met_renewals),
    "part au-dessus de la ligne directrice de l'année (%)": round(
        (the_met_renewals["contract_rent_growth_pct"]
         > the_met_renewals["lease_start_year"].map(regulatory_rates["ontario_guideline_pct"])).mean() * PCT, 1),
}).to_frame("valeur")

**Lecture (Ontario) :** The Met génère ses premiers baux en 2023, bien après le 15 novembre 2018 : il est **vraisemblablement
exempté** de la ligne directrice. Les données vont dans ce sens : **81 % de ses renouvellements dépassent la ligne directrice de
l'année** (2,5 %), et la hausse contractuelle médiane des renouvellements atteint 6,3 % en 2025, ce qui serait impossible pour un
logement assujetti. Le statut exact reste à confirmer avec un document (date de première occupation).

**Conséquences pour la prévision :**
- les renouvellements et les relocations sont **séparés** dans le modèle (`is_renewal`), et chaque immeuble garde sa propre part
  de renouvellements;
- le taux réglementaire est **propre à chaque province** : TAL pour les immeubles québécois, ligne directrice pour The Met;
  il est fourni comme variable, **sans plafond**, puisque The Met est probablement exempté et que le TAL n'est qu'indicatif;
- The Met n'a des paires qu'à partir de 2024 : sa prévision repose en partie sur ce que le modèle apprend du Québec (section 8).

Écart entre types de baux : en 2022-2023 les renouvellements montent plus vite, en 2024-2025 ce sont les relocations
(loyers de marché en hausse), ce qui justifie de ne pas les fusionner sans le dire au modèle.

## 6. Données publiques : intégrer et réconcilier

| Source | Série | Marché | Années | Usage |
|---|---|---|---|---|
| TAL, gouvernement de l'Ontario | taux annuel d'augmentation (logement non chauffé) / ligne directrice | Québec / Ontario | 2019-2026 | variable du modèle (5a) |
| SCHL, Enquête sur les logements locatifs (fichiers Excel 2021-2025) | variation du loyer des 2 chambres, **échantillon fixe** | RMR Montréal, Ottawa (partie Ontario) | 2020-2025 | réconciliation (6b) |
| idem | taux d'inoccupation | idem | 2021-2025 | concessions (6c), variable testée |
| Statistique Canada (18-10-0004) | IPC, composante **logement** | Canada | 2021-2025 | réconciliation (6b), variable testée |
| Statistique Canada (14-10-0460, 17-10-0149) | chômage; immigration et résidents non permanents | RMR | 2021-2025 | variables testées (7e) |

L'échantillon fixe de la SCHL compare **les mêmes logements** d'une année à l'autre : c'est l'équivalent marché de notre mesure
à unité constante, en loyer contractuel. On lit d'abord les séries, puis on les rapproche de la tendance interne.

### 6a. Chargement des séries publiques

In [ ]:
CMHC_MARKET = {QUEBEC: "Montréal", ONTARIO: "Ottawa — partie Ontario"}
CPI_GEO = "Canada"                     # IPC.csv ne contient que le Canada
CPI_COMPONENT = "Logement"             # composante la plus proche du loyer dans IPC.csv
LABOUR_GEO = {QUEBEC: "Montréal, Québec", ONTARIO: "Toronto, Ontario"}     # Ottawa absent du fichier : Toronto en substitut
DEMOGRAPHY_GEO = {QUEBEC: "Montréal (CMA), Quebec", ONTARIO: "Ottawa - Gatineau (CMA), Ontario part, Ontario"}
IMMIGRATION_COMPONENTS = ["Immigrants", "Net non-permanent residents"]
CMHC_FIXED_SAMPLE_COLUMNS = {"previous": "P", "current": "R"}   # tableau 1.0 : variation oct. t-2 -> t-1 et t-1 -> t
CMHC_FIRST_FILE_YEAR, CMHC_LAST_FILE_YEAR = 2021, LAST_OBSERVED_YEAR
RENTAL_UNIVERSE_REFERENCE_YEAR = 2021   # parc locatif SCHL servant à rapporter l'immigration à la taille du marché
PROVINCES = [QUEBEC, ONTARIO]


def cmhc_fixed_sample_growth():
    """Variation du loyer des 2 chambres, échantillon fixe (SCHL, tableau 1.0), par marché et année d'enquête."""
    records = []
    for file_year in range(CMHC_FIRST_FILE_YEAR, CMHC_LAST_FILE_YEAR + 1):
        sheets = read_workbook(RAW_PUBLIC_DIR / f"rmr-canada-{file_year}-fr.xlsx")
        table = next(name for name in sheets if name.replace("Tableau", "Table") == "Table 1.0")
        for _, cells in sheets[table]:
            if cells.get("A") in CMHC_MARKET_LABELS:
                market = CMHC_MARKET_LABELS[cells["A"]]
                records.append({"market": market, "year": file_year, "growth": number(cells.get(CMHC_FIXED_SAMPLE_COLUMNS["current"]))})
                records.append({"market": market, "year": file_year - 1, "growth": number(cells.get(CMHC_FIXED_SAMPLE_COLUMNS["previous"]))})
    return pd.DataFrame(records).drop_duplicates(["market", "year"]).set_index(["market", "year"])["growth"]


def statcan_series(path, filters, geo_column="GÉO"):
    """Série StatCan (format CSV de la table) filtrée, indexée par (géographie, période)."""
    table = pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig")
    for column, value in filters.items():
        table = table[table[column] == value]
    period_column = table.columns[0]
    value_column = "VALEUR" if "VALEUR" in table.columns else "VALUE"
    return table.set_index([geo_column, period_column])[value_column].astype(float)


cmhc_survey, _ = extract_public_data(RAW_PUBLIC_DIR)
cmhc_survey = cmhc_survey.set_index(["market", "year"])
fixed_sample_growth = cmhc_fixed_sample_growth()
cpi_shelter = statcan_series(CPI_PATH, {"Produits et groupes de produits": CPI_COMPONENT})
unemployment = statcan_series(LABOUR_PATH, {"Caractéristiques de la population active": "Taux de chômage"})
demography = pd.read_csv(DEMOGRAPHY_PATH)
immigration = (demography[demography["Components of population growth"].isin(IMMIGRATION_COMPONENTS)]
               .groupby(["GEO", "REF_DATE"])["VALUE"].sum())


def annual_mean(monthly_series, geo):
    """Moyenne annuelle d'une série mensuelle (années complètes seulement)."""
    values = monthly_series.xs(geo)
    years = pd.PeriodIndex(values.index, freq="M").year
    by_year = values.groupby(years)
    return by_year.mean()[by_year.size() == len(range(1, 13))]


signal_rows = []
for province in PROVINCES:
    cpi_shelter_by_year = annual_mean(cpi_shelter, CPI_GEO)
    unemployment_by_year = annual_mean(unemployment, LABOUR_GEO[province])
    rental_universe = cmhc_survey.loc[(CMHC_MARKET[province], RENTAL_UNIVERSE_REFERENCE_YEAR), "rental_universe"]
    for year in range(FIRST_PUBLIC_YEAR, FORECAST_YEAR + 1):
        regulatory_column = "tal_unheated_pct" if province == QUEBEC else "ontario_guideline_pct"
        signal_rows.append({
            "province": province, "year": year,
            "regulatory_rate": regulatory_rates[regulatory_column].get(year, np.nan),
            "cmhc_fixed_sample_growth": fixed_sample_growth.get((CMHC_MARKET[province], year), np.nan),
            "cpi_shelter_growth": (cpi_shelter_by_year.get(year, np.nan) / cpi_shelter_by_year.get(year - 1, np.nan) - 1) * PCT,
            "schl_vacancy": cmhc_survey["vacancy_rate"].get((CMHC_MARKET[province], year), np.nan),
            "unemployment_rate": unemployment_by_year.get(year, np.nan),
            # année démographique juillet (t-1) à juin t, rapportée au parc locatif du marché
            "immigration_per_100_rental_units":
                immigration.get((DEMOGRAPHY_GEO[province], f"{year - 1}/{year}"), np.nan) / rental_universe * PCT,
        })
market_signals = pd.DataFrame(signal_rows).set_index(["province", "year"]).sort_index()
market_signals.loc[(slice(None), slice(FIRST_ANALYSIS_YEAR, FORECAST_YEAR)), :].round(2)

### 6b. Réconcilier la tendance interne avec le marché

On compare, par marché, la hausse **contractuelle** d'Équinoxe à unité constante avec l'échantillon fixe de la SCHL (une mesure
du loyer **inscrit**, sans concessions, sur les mêmes logements) et avec l'IPC logement du Canada (contexte d'inflation), puis la
hausse **effective** d'Équinoxe.

Les années ne sont pas tout à fait alignées : la SCHL mesure d'octobre à octobre, l'IPC en moyenne annuelle, nous par année de
début de bail. On compare donc des **ordres de grandeur et des tendances**, pas des chiffres au dixième.

In [ ]:
reconciliation_rows = []
for province in PROVINCES:
    provincial_pairs = pairs[pairs["sState"] == province]
    for year in EXTENDED_BACKTEST_YEARS:
        year_pairs = provincial_pairs[provincial_pairs["lease_start_year"] == year]
        reconciliation_rows.append({
            "province": province, "year": year, "pairs": len(year_pairs),
            "equinoxe_contract": year_pairs["contract_rent_growth_pct"].median(),
            "cmhc_fixed_sample": market_signals.loc[(province, year), "cmhc_fixed_sample_growth"],
            "cpi_shelter_canada": market_signals.loc[(province, year), "cpi_shelter_growth"],
            "regulatory_rate": market_signals.loc[(province, year), "regulatory_rate"],
            "equinoxe_effective": year_pairs["effective_rent_growth_pct"].median(),
        })
reconciliation = pd.DataFrame(reconciliation_rows).set_index(["province", "year"])
print(reconciliation.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, province in zip(axes, PROVINCES):
    table = reconciliation.xs(province)
    ax.plot(table.index, table["equinoxe_contract"], "o-", lw=2, label="Équinoxe, contractuel (même unité)")
    ax.plot(table.index, table["equinoxe_effective"], "s-", lw=2, label="Équinoxe, effectif (même unité)")
    ax.plot(table.index, table["cmhc_fixed_sample"], "^--", label="SCHL, échantillon fixe 2 ch.")
    ax.plot(table.index, table["cpi_shelter_canada"], "d--", label="IPC logement, Canada")
    ax.step(table.index, table["regulatory_rate"], where="mid", color="grey", label="TAL / ligne directrice")
    ax.set(title=f"{CMHC_MARKET[province]}", xlabel="année", ylabel="croissance annuelle (%)", xticks=EXTENDED_BACKTEST_YEARS)
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Lecture :**

- **Québec, loyer contractuel : Équinoxe suit le marché, avec un décalage.** En 2022 et en 2025, la hausse contractuelle
  d'Équinoxe est au niveau de l'échantillon fixe de la SCHL (5,4 % contre 5,4 % en 2022; 7,1 % contre 7,2 % en 2025). En 2023-2024,
  le marché accélère (7,9 % et 6,3 %) mais Équinoxe ralentit (2,3 % et 3,4 %), puis rattrape en 2025.
- **Le creux de 2023 est propre à Équinoxe** : le marché accélérait. Aucune série publique ne pouvait donc l'annoncer, ce qui
  explique l'erreur du backtest 2023 (section 7d).
- **Loyer effectif : l'écart avec le marché, ce sont les concessions.** En 2024-2025, Équinoxe affiche des hausses contractuelles
  qui rejoignent le marché mais en redonne une partie : la hausse effective (1,8 % puis 3,6 %) reste nettement sous le marché.
- **Ontario :** The Met dépasse le marché d'Ottawa en contractuel en 2025 (6,9 % contre 3,4 % pour la SCHL), cohérent avec un
  immeuble neuf exempté du plafond; en effectif (3,4 %), il rejoint le marché.
- **IPC logement (Canada) :** l'inflation du logement ralentit, de 6,9 % en 2022 à 3,1 % en 2025. C'est un contexte national et
  plus large que le loyer (il inclut l'intérêt hypothécaire et l'énergie) : il ne se compare pas directement à nos hausses.

**Conséquence :** la tendance interne et le marché sont **réconciliés** : l'écart vient de deux mécanismes identifiés, le décalage
de 2023-2024 et les concessions. Pour 2026, la question n'est donc pas seulement « le marché va-t-il ralentir? », mais surtout
« les concessions vont-elles continuer de grossir? ».



### 6c. Les concessions répondent-elles au marché?

La section 4 a montré que les concessions sont une politique globale qui dépend de l'année. Hypothèse : elles suivent le
**taux d'inoccupation du marché**. On compare la hausse médiane du rabais (Québec) et la variation de l'inoccupation à Montréal.

In [ ]:
quebec_pairs = pairs[pairs["sState"] == QUEBEC]
vacancy_montreal = market_signals.xs(QUEBEC)["schl_vacancy"]
discount_vs_vacancy = pd.DataFrame({
    "median_discount_change_pts": median_by_year(quebec_pairs, "discount_change_pts"),
    "vacancy_rate": vacancy_montreal,
    "vacancy_change_pts": vacancy_montreal.diff(),
}).loc[analysis_years].dropna()   # inoccupation SCHL disponible depuis 2021, variation depuis 2022
vacancy_correlation = discount_vs_vacancy[["median_discount_change_pts", "vacancy_change_pts"]].corr().iloc[0, 1]
print(discount_vs_vacancy.round(2))
print(f"\nCorrélation hausse du rabais / variation de l'inoccupation (même année) : {vacancy_correlation:.2f} "
      f"(n = {len(discount_vs_vacancy)})")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(discount_vs_vacancy["vacancy_change_pts"], discount_vs_vacancy["median_discount_change_pts"])
for year, row in discount_vs_vacancy.iterrows():
    ax.annotate(str(year), (row["vacancy_change_pts"], row["median_discount_change_pts"]), textcoords="offset points", xytext=(5, 5))
ax.set(xlabel="variation du taux d'inoccupation, Montréal (points, SCHL)", ylabel="hausse médiane du rabais (points)",
       title="Inoccupation du marché et concessions d'Équinoxe")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Lecture :** l'inoccupation à Montréal baisse jusqu'en 2023 (1,5 %, marché très serré), puis remonte (2,1 % en 2024,
2,9 % en 2025). Les rabais d'Équinoxe suivent : stables quand le marché se resserre, en forte hausse quand il se détend
(corrélation de 0,79). C'est cohérent avec l'hypothèse : **les concessions répondent au marché**.

**Prudence :** 4 années seulement, et la relation est **contemporaine** : l'inoccupation d'octobre *t* n'est connue qu'à la fin
de *t*. Elle ne peut donc pas servir directement à prévoir *t*; on teste dans le modèle l'inoccupation de *t* - 1 (7e).



**Comment ces données entrent dans la prévision (section 7) :**
- le **taux réglementaire** de l'année (connu avant l'année) est une variable du modèle, propre à chaque province;
- les autres séries ne sont connues qu'**avec retard** : une prévision de l'année *t* ne peut utiliser que les valeurs de *t* - 1.
  On teste chacune avec ce décalage, et on ne la garde que si elle améliore le backtest (7e);
- la réconciliation (6b) sert aussi à **vérifier la prévision 2026** contre le marché (7g).

## 7. Prévoir et valider

**Règle du backtest :** pour prévoir l'année *t*, la méthode n'utilise que ce qui était **connu au début de *t*** :
les paires dont le nouveau bail a débuté avant *t*, les loyers affichés avant le 1er janvier de *t*, le taux réglementaire de *t*
(publié avant l'année) et les séries publiques de *t* - 1. On compare ensuite à la médiane réellement observée en *t*.

### 7a. Point de départ : extrapolations de tendance

Trois règles simples, appliquées à la série de la cible (3c) : reprendre la hausse de l'année précédente, la moyenne des trois
dernières années, ou prolonger une tendance linéaire.

In [ ]:
TREND_WINDOW_YEARS = 3


def trend_forecasts(target_series, year):
    """Extrapolations de la cible pour `year`, à partir des seules années antérieures."""
    history = target_series[target_series.index < year].dropna()
    slope, intercept = np.polyfit(history.index, history.to_numpy(), deg=1)
    return {
        "hausse de l'an dernier": history.iloc[-1],
        f"moyenne des {TREND_WINDOW_YEARS} dernières années": history.iloc[-TREND_WINDOW_YEARS:].mean(),
        "tendance linéaire": slope * year + intercept,
    }


baseline_forecasts = pd.DataFrame({year: trend_forecasts(yearly_target, year) for year in EXTENDED_BACKTEST_YEARS}).T
baseline_errors = baseline_forecasts.sub(yearly_target.loc[EXTENDED_BACKTEST_YEARS], axis=0)
baseline_summary = pd.DataFrame({
    "MAE 2023-2025": baseline_errors.loc[REQUIRED_BACKTEST_YEARS].abs().mean(),
    "MAE 2021-2025": baseline_errors.abs().mean(),
    f"prévision {FORECAST_YEAR}": pd.Series(trend_forecasts(yearly_target, FORECAST_YEAR)),
})
baseline_summary.round(2)

**Lecture :** ces règles se trompent d'environ 1,3 à 1,5 point par an. Elles ne savent rien des appartements loués ni des
concessions : elles servent de **référence à battre**.

### 7b. Le modèle : gradient boosting au niveau des paires de baux

Plutôt que de prévoir une série de 7 points annuels, on suit littéralement la définition : le modèle prédit **la hausse de chaque
bail** à partir de ce qu'on sait avant ce bail, puis on prend la **médiane**. Il apprend ainsi sur environ 3 000 paires.

Un **gradient boosting** (scikit-learn, `HistGradientBoostingRegressor`) est justifié parce que les effets ne sont pas linéaires et
interagissent (un gros rabais précédent ne recule pas de la même façon pour un renouvellement et une relocation), et qu'il gère
les valeurs manquantes. Il est contraint pour peu de données : erreur absolue (il vise la médiane), arbres de profondeur 3,
au moins 40 paires par feuille.

| Variable | Pourquoi | Connue avant le bail? |
|---|---|---|
| immeuble (indicatrices), chambres, superficie, étage | le produit loué | oui |
| `gap_years` | durée écoulée depuis le bail précédent | oui (fin du bail précédent) |
| `is_renewal` | renouvellement ou relocation (section 5) | **non** : voir ci-dessous |
| `prev_discount_pct` | rabais du bail précédent : un gros rabais peut reculer (section 4) | oui |
| `regulatory_rate` | TAL ou ligne directrice de l'année, selon la province (section 5) | oui, publié avant l'année |
| `asking_gap_pct` | écart entre le dernier loyer affiché du logement avant le 1er janvier et le loyer contractuel du bail précédent | oui |

**Le type du prochain bail n'est pas connu d'avance** : chaque bail est prédit comme renouvellement **et** comme relocation,
pondéré par la part historique de renouvellements de son immeuble. **Population 2026** : le prochain bail de chaque logement
débute le lendemain de la fin du dernier bail connu; on retient ceux qui débutent en 2026, avec les mêmes filtres d'écart.

### 7c. `estimate_2026()` et `backtest()`

In [ ]:
RETAINED_OPTIONS = {"prev_discount": True, "regulatory_rate": True, "asking_gap": True, "lagged_public_signals": ()}


def market_signal(df, signals, column, years_before=0):
    """Valeur d'une série publique pour la province de chaque bail, `years_before` ans avant l'année du bail."""
    keys = pd.MultiIndex.from_arrays([df["sState"], df["lease_start_year"] - years_before])
    return signals[column].reindex(keys).to_numpy()


def asking_gap_pct(df, asking_history):
    """Écart (%, log) entre le dernier loyer affiché avant le 1er janvier de l'année du bail et le loyer contractuel précédent."""
    query = pd.DataFrame({
        "row": np.arange(len(df)),
        "sPropCode": df["sPropCode"].to_numpy(), "sUnitCode": df["sUnitCode"].to_numpy(),
        "cutoff": pd.to_datetime(df["lease_start_year"].astype(str) + "-01-01").to_numpy(),
    }).sort_values("cutoff")
    history = asking_history[UNIT_KEY + ["ask_month", "sAskingRent"]].sort_values("ask_month")
    matched = pd.merge_asof(query, history, left_on="cutoff", right_on="ask_month", by=UNIT_KEY, allow_exact_matches=False)
    last_asking_rent = matched.sort_values("row")["sAskingRent"].to_numpy()
    return np.log(last_asking_rent / df["prev_contract_rent"].to_numpy()) * PCT


def model_features(df, context, options):
    features = df[UNIT_FEATURES].astype(float).copy()
    if options["prev_discount"]:
        features["prev_discount_pct"] = df["prev_discount_pct"]
    if options["regulatory_rate"]:
        features["regulatory_rate"] = market_signal(df, context["signals"], "regulatory_rate")
    if options["asking_gap"]:
        features["asking_gap_pct"] = asking_gap_pct(df, context["asking"])
    for signal in options["lagged_public_signals"]:
        features[f"{signal}_prev_year"] = market_signal(df, context["signals"], signal, years_before=1)
    for building in context["buildings"]:
        features[f"building_{building}"] = (df["sBuilding"] == building).astype(float)
    return features


def fit_pair_model(training_pairs, context, options):
    features = model_features(training_pairs, context, options)
    features = features.loc[:, features.notna().any()]   # une série publique sans historique est ignorée
    return HistGradientBoostingRegressor(**GBM_PARAMS).fit(features, training_pairs["effective_rent_growth_pct"])


def predict_pairs(model, df, context, options):
    return model.predict(model_features(df, context, options)[model.feature_names_in_])


def weighted_median(values, weights):
    order = np.argsort(values)
    sorted_values, cumulative_weights = values[order], np.cumsum(weights[order])
    return sorted_values[np.searchsorted(cumulative_weights, cumulative_weights[-1] / 2)]


def predict_both_lease_types(model, population, training_pairs, context, options):
    """Prédit chaque bail comme renouvellement et comme relocation, pondéré par la part de renouvellements de l'immeuble."""
    renewal_share = (population["sBuilding"].map(training_pairs.groupby("sBuilding")["is_renewal"].mean())
                     .fillna(training_pairs["is_renewal"].mean()).to_numpy())
    scenarios = []
    for lease_type, weight in [(RENEWAL, renewal_share), (TURNOVER, 1 - renewal_share)]:
        scenario = population.assign(is_renewal=lease_type)
        scenario["predicted_growth"] = predict_pairs(model, scenario, context, options)
        scenario["weight"] = weight
        scenarios.append(scenario)
    return pd.concat(scenarios)


def next_lease_population(lease_df, year):
    """Logements dont le prochain bail débute en `year` : le lendemain de la fin du dernier bail connu."""
    last_leases = lease_df.sort_values("lease_start").groupby(UNIT_KEY).tail(1)
    next_start = last_leases["lease_end"] + pd.Timedelta(days=1)
    population = last_leases.assign(
        lease_start_year=next_start.dt.year,
        gap_years=(next_start - last_leases["lease_start"]).dt.days / DAYS_PER_YEAR,
        prev_contract_rent=last_leases["contract_rent"],
        prev_discount_pct=discount_pct(last_leases["effective_rent"], last_leases["contract_rent"]))
    has_comparable_gap = population["gap_years"].between(MIN_GAP_YEARS, MAX_GAP_YEARS, inclusive="neither")
    return population[(population["lease_start_year"] == year) & has_comparable_gap]


def make_context(lease_df, asking_history, external):
    return {"asking": asking_history, "signals": external, "buildings": sorted(lease_df["sBuilding"].unique())}


def forecast_year(lease_df, year, population, context, options):
    """Entraîne sur les paires antérieures à `year`, prédit chaque bail de `population`, renvoie (médiane pondérée, prédictions, modèle)."""
    history = build_comparable_pairs(lease_df)
    training_pairs = history[history["lease_start_year"].between(FIRST_ANALYSIS_YEAR, year - 1)]
    model = fit_pair_model(training_pairs, context, options)
    predictions = predict_both_lease_types(model, population, training_pairs, context, options)
    growth = weighted_median(predictions["predicted_growth"].to_numpy(), predictions["weight"].to_numpy())
    return growth, predictions, model


def estimate_2026(leases, asking, external=None, options=RETAINED_OPTIONS):
    """Hausse 2026 (%) du loyer effectif à unité constante, médiane de tous les baux qui débutent en 2026 (section 1).

    `external` : séries publiques par province et année (par défaut `market_signals`, section 6a).
    """
    external = market_signals if external is None else external
    context = make_context(leases, asking, external)
    growth, _, _ = forecast_year(leases, FORECAST_YEAR, next_lease_population(leases, FORECAST_YEAR), context, options)
    return growth


def backtest(leases, target_year, asking=None, external=None, options=RETAINED_OPTIONS):
    """Applique la méthode comme à la fin de target_year - 1, puis compare à la hausse réellement observée en target_year.

    La population est celle des baux qui ont réellement débuté en target_year; on n'utilise que leur bail précédent,
    le logement et l'information publique disponible au 1er janvier.
    """
    asking = asking_history if asking is None else asking
    external = market_signals if external is None else external
    context = make_context(leases, asking, external)
    history = build_comparable_pairs(leases)
    population = history[history["lease_start_year"] == target_year]
    forecast, _, _ = forecast_year(leases, target_year, population, context, options)
    actual = population["effective_rent_growth_pct"].median()
    return {"forecast": float(forecast), "actual": float(actual), "error": float(forecast - actual)}


def lease_level_error(leases, asking, external, options=RETAINED_OPTIONS, years=EXTENDED_BACKTEST_YEARS):
    """Erreur absolue médiane par bail (type de bail réel), hors période, moyennée sur les années de backtest (points)."""
    context = make_context(leases, asking, external)
    history = build_comparable_pairs(leases)
    yearly_errors = []
    for year in years:
        training_pairs = history[history["lease_start_year"].between(FIRST_ANALYSIS_YEAR, year - 1)]
        test_pairs = history[history["lease_start_year"] == year]
        model = fit_pair_model(training_pairs, context, options)
        yearly_errors.append(np.median(np.abs(predict_pairs(model, test_pairs, context, options)
                                              - test_pairs["effective_rent_growth_pct"])))
    return np.mean(yearly_errors)


asking_history = asking

### 7d. Backtest

**Deux critères :**
- **MAE annuelle** : erreur absolue moyenne de la médiane prévue, sur 2023-2025 (exigé) et sur 2021-2025 (plus robuste);
- **erreur par bail** : erreur absolue médiane entre la hausse prévue et la hausse réelle de chaque bail. Elle repose sur environ
  2 600 baux plutôt que sur 5 médianes, et sert à vérifier qu'un gain annuel n'est pas dû au hasard.

In [ ]:
for year in REQUIRED_BACKTEST_YEARS:   # vérification suggérée par le notebook de départ
    print(year, {key: round(value, 2) for key, value in backtest(leases, year).items()})

backtest_results = pd.DataFrame({year: backtest(leases, year) for year in EXTENDED_BACKTEST_YEARS}).T
model_mae_required = backtest_results.loc[REQUIRED_BACKTEST_YEARS, "error"].abs().mean()
model_mae_extended = backtest_results["error"].abs().mean()
model_lease_error = lease_level_error(leases, asking, market_signals)

backtest_comparison = pd.concat([
    baseline_summary[["MAE 2023-2025", "MAE 2021-2025"]],
    pd.DataFrame({"MAE 2023-2025": [model_mae_required], "MAE 2021-2025": [model_mae_extended]},
                 index=["modèle (gradient boosting)"]),
])
print()
print(backtest_results.round(2), end="\n\n")
print(backtest_comparison.round(2), end="\n\n")
print(f"Erreur médiane par bail du modèle : {model_lease_error:.2f} points")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(yearly_target.index, yearly_target, "k-o", lw=2.5, label="réel (cible)")
ax.plot(backtest_results.index, backtest_results["forecast"], "--s", lw=2, label="modèle")
for column, style in zip(baseline_forecasts.columns, [":^", ":v", ":d"]):
    ax.plot(baseline_forecasts.index, baseline_forecasts[column], style, alpha=0.6, label=column)
ax.axvspan(REQUIRED_BACKTEST_YEARS[0] - 0.5, REQUIRED_BACKTEST_YEARS[-1] + 0.5, color="grey", alpha=0.1,
           label="backtest exigé")
ax.set(xlabel="année", ylabel="croissance du loyer effectif, même unité (%)",
       title="Backtest : prévisions faites avec l'information disponible l'année précédente")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Lecture :** 

- **Le modèle bat les extrapolations de tendance** sur les deux périodes : MAE de 1,11 point sur 2023-2025 (contre 1,36 pour la
  meilleure règle simple) et 1,04 point sur 2021-2025 (contre 1,24).
- **2023 reste la grande erreur** (4,3 % prévu pour 2,0 % réel) : c'est le ralentissement propre à Équinoxe, qu'aucune information
  disponible fin 2022 n'annonçait (6b). 2024 et 2025 sont prévus à moins de 0,7 point.
- L'erreur par bail (2,88 points) mesure la difficulté de prévoir **chaque** bail; la médiane du portefeuille est beaucoup plus
  stable, d'où une erreur annuelle d'environ 1 point.

### 7e. Quelles variables comptent? Données publiques testées dans le modèle

On refait le backtest en retirant ou en ajoutant **un groupe de variables à la fois**. Une variable n'est retenue que si elle améliore
**les deux critères** et si son effet a un sens économique.

In [ ]:
VARIANT_OPTIONS = {
    "retenu": RETAINED_OPTIONS,
    "sans loyer affiché": {**RETAINED_OPTIONS, "asking_gap": False},
    "sans rabais précédent": {**RETAINED_OPTIONS, "prev_discount": False},
    "sans taux réglementaire": {**RETAINED_OPTIONS, "regulatory_rate": False},
    "+ inoccupation SCHL (t-1)": {**RETAINED_OPTIONS, "lagged_public_signals": ("schl_vacancy",)},
    "+ IPC logement (t-1)": {**RETAINED_OPTIONS, "lagged_public_signals": ("cpi_shelter_growth",)},
    "+ chômage (t-1)": {**RETAINED_OPTIONS, "lagged_public_signals": ("unemployment_rate",)},
    "+ immigration (t-1)": {**RETAINED_OPTIONS, "lagged_public_signals": ("immigration_per_100_rental_units",)},
}

variant_rows = {}
for variant, options in VARIANT_OPTIONS.items():
    forecasts = {year: backtest(leases, year, options=options)["forecast"] for year in EXTENDED_BACKTEST_YEARS}
    errors = pd.Series(forecasts) - yearly_target.loc[EXTENDED_BACKTEST_YEARS]
    variant_rows[variant] = {
        "MAE 2023-2025": errors.loc[REQUIRED_BACKTEST_YEARS].abs().mean(),
        "MAE 2021-2025": errors.abs().mean(),
        "erreur par bail": lease_level_error(leases, asking, market_signals, options),
        f"prévision {FORECAST_YEAR}": estimate_2026(leases, asking, options=options),
    }
variants = pd.DataFrame(variant_rows).T
variants.round(2)

**Lecture :** 

- **Le rabais du bail précédent est la variable la plus utile** : sans elle, l'erreur par bail passe de 2,88 à 3,26 points et la
  MAE 2023-2025 de 1,11 à 1,49. C'est aussi elle qui porte la prévision 2026 (2,3 % sans elle).
- **Le loyer affiché** améliore les deux critères (erreur par bail 2,95 → 2,88; MAE 2023-2025 1,25 → 1,11) : retenu.
- **Le taux réglementaire** améliore la prévision de chaque bail (3,02 → 2,88) mais pas la MAE 2021-2025 : comme il ne prend qu'une
  valeur par année et par province, les arbres s'en servent en partie pour reconnaître l'année. Retenu, parce qu'il est utile bail
  par bail, qu'il est connu d'avance et qu'il a un sens économique.
- **Séries publiques (décalées d'un an) : aucune n'est retenue.** Elles commencent en 2021 : avec le décalage, le modèle
  n'en dispose qu'à partir des baux de 2022 (inoccupation, chômage) ou 2023 (IPC, immigration). Leur effet n'est donc testable
  que sur les backtests 2024-2025, avec deux ou trois années pour l'apprendre.
  - aucune n'améliore l'erreur par bail (2,89 à 2,90 contre 2,88);
  - l'**inoccupation SCHL** gagne un peu en MAE annuelle mais fait bondir 2026 à 5,0 % : l'inoccupation d'octobre 2025 (2,9 %
    à Montréal) dépasse tout ce que le modèle a vu. Gain négligeable, extrapolation forte;
  - le **chômage** gagne aussi un peu en MAE annuelle, mais dégrade l'erreur par bail et déplace 2026 à 3,8 %;
  - l'**IPC logement** et l'**immigration** n'apportent rien et déplacent 2026 vers le bas (environ 3,2 %).

  Ces séries servent donc de **contexte et de vérification** (6b, 6c, 7g), pas de variables du modèle.



**Modèle retenu :** logement, immeuble, type de bail, rabais précédent, taux réglementaire provincial et loyer affiché.

### 7f. Prévision 2026

Le modèle final est entraîné sur toutes les paires jusqu'en 2025, avec les taux réglementaires 2026 (TAL : 3,1 %; Ontario : 2,1 %)
et les loyers affichés jusqu'en décembre 2025, puis sauvegardé (livrable « modèle entraîné »).

**Fourchette et scénarios :**
- **bas / haut** : la prévision ± l'erreur typique du backtest (MAE 2021-2025);
- **sensibilité aux choix de modélisation** : l'étendue des variantes de 7e dont l'erreur par bail reste à moins de 0,1 point du modèle retenu.

In [ ]:
population_2026 = next_lease_population(leases, FORECAST_YEAR)
final_context = make_context(leases, asking, market_signals)
forecast_2026, predictions_2026, final_model = forecast_year(leases, FORECAST_YEAR, population_2026, final_context, RETAINED_OPTIONS)
assert np.isclose(forecast_2026, estimate_2026(leases, asking))

comparable_variants = variants[variants["erreur par bail"] <= variants.loc["retenu", "erreur par bail"] + COMPARABLE_VARIANT_TOLERANCE]
scenarios_2026 = pd.Series({
    "bas (prévision - MAE)": forecast_2026 - model_mae_extended,
    "central": forecast_2026,
    "haut (prévision + MAE)": forecast_2026 + model_mae_extended,
})
print(f"{len(population_2026):,} logements dont le prochain bail débute en {FORECAST_YEAR}")
print(f"Rabais moyen au bail précédent : population {FORECAST_YEAR} = {population_2026['prev_discount_pct'].mean():.1f} %, "
      f"baux de {LAST_OBSERVED_YEAR} = {pairs.loc[pairs['lease_start_year'] == LAST_OBSERVED_YEAR, 'prev_discount_pct'].mean():.1f} %\n")
print(f"Hausse {FORECAST_YEAR} prévue (loyer effectif, même unité, médiane) : {forecast_2026:.2f} %")
print(scenarios_2026.round(2).to_string())
print(f"variantes comparables ({len(comparable_variants)}) : {comparable_variants[f'prévision {FORECAST_YEAR}'].min():.2f} % "
      f"à {comparable_variants[f'prévision {FORECAST_YEAR}'].max():.2f} %")

MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump({"model": final_model, "options": RETAINED_OPTIONS, "buildings": final_context["buildings"],
             "features": list(final_model.feature_names_in_)}, MODEL_PATH)
print(f"\nModèle sauvegardé : {MODEL_PATH}")

### Par immeuble et par nombre de chambres (bonus)

Chaque bail apparaît deux fois dans les prédictions (renouvellement et relocation, pondérés) : on prend la médiane pondérée par groupe.

In [ ]:
def forecast_by(predictions, group_column):
    rows = {}
    for group, group_predictions in predictions.groupby(group_column):
        rows[group] = {
            f"baux_{FORECAST_YEAR}": group_predictions.index.nunique(),
            "hausse_effective_prevue_pct": weighted_median(group_predictions["predicted_growth"].to_numpy(),
                                                           group_predictions["weight"].to_numpy()),
        }
    return pd.DataFrame(rows).T


forecast_by_building = forecast_by(predictions_2026, "sBuilding").sort_values("hausse_effective_prevue_pct")
forecast_by_bedrooms = forecast_by(predictions_2026, "sBeds")
print("Par immeuble :")
print(forecast_by_building.round(2), end="\n\n")
print("Par nombre de chambres :")
print(forecast_by_bedrooms.round(2))

### Tableau de bord de la prévision (bonus)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), gridspec_kw={"width_ratios": [1.3, 1.2, 1]})

history_and_forecast = yearly_target.copy()
axes[0].plot(history_and_forecast.index, history_and_forecast, "k-o", lw=2, label="réel")
axes[0].errorbar([FORECAST_YEAR], [forecast_2026], yerr=[[forecast_2026 - scenarios_2026.iloc[0]], [scenarios_2026.iloc[-1] - forecast_2026]],
                 fmt="s", color="tab:red", capsize=6, label=f"prévision {FORECAST_YEAR} (± MAE)")
axes[0].set(title="Hausse du loyer effectif, même unité", xlabel="année", ylabel="%")
axes[0].legend(fontsize=8)

axes[1].barh(forecast_by_building.index, forecast_by_building["hausse_effective_prevue_pct"], color="tab:blue")
axes[1].axvline(forecast_2026, color="tab:red", ls="--", label="portefeuille")
for position, (building, row) in enumerate(forecast_by_building.iterrows()):
    axes[1].annotate(f"{row['hausse_effective_prevue_pct']:.1f} % ({row[f'baux_{FORECAST_YEAR}']:.0f} baux)",
                     (0, position), xytext=(4, -3), textcoords="offset points", fontsize=8, color="white")
axes[1].set(title=f"Prévision {FORECAST_YEAR} par immeuble", xlabel="%")
axes[1].legend(fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))

axes[2].bar(forecast_by_bedrooms.index.astype(int).astype(str), forecast_by_bedrooms["hausse_effective_prevue_pct"], color="tab:green")
axes[2].axhline(forecast_2026, color="tab:red", ls="--")
axes[2].set(title=f"Prévision {FORECAST_YEAR} par nombre de chambres", xlabel="chambres", ylabel="%")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 7g. Vérification de la prévision 2026 contre le marché

On met la prévision à côté des signaux disponibles fin 2025. Les signaux de marché portent sur le loyer **contractuel**; notre
prévision porte sur le loyer **effectif**. L'écart attendu entre les deux est celui des concessions (section 4).

In [ ]:
last_lease_rent = leases.sort_values("lease_start").groupby(UNIT_KEY).tail(1).set_index(UNIT_KEY)["contract_rent"]
proposed_markup_pct = (listings.set_index(UNIT_KEY)["proposed_next_rent"] / last_lease_rent.reindex(listings.set_index(UNIT_KEY).index) - 1) * PCT

market_check_2026 = pd.Series({
    f"prévision {FORECAST_YEAR}, loyer effectif (modèle)": forecast_2026,
    f"hausse contractuelle Équinoxe {LAST_OBSERVED_YEAR}": candidate_definitions.loc[LAST_OBSERVED_YEAR, "same_unit_contract"],
    f"hausse effective Équinoxe {LAST_OBSERVED_YEAR}": yearly_target.loc[LAST_OBSERVED_YEAR],
    f"loyer proposé (listings) vs dernier bail, médiane": proposed_markup_pct.median(),
    f"TAL {FORECAST_YEAR} (renouvellements Québec)": market_signals.loc[(QUEBEC, FORECAST_YEAR), "regulatory_rate"],
    f"ligne directrice Ontario {FORECAST_YEAR} (The Met probablement exempté)": market_signals.loc[(ONTARIO, FORECAST_YEAR), "regulatory_rate"],
    f"SCHL échantillon fixe Montréal {LAST_OBSERVED_YEAR}": market_signals.loc[(QUEBEC, LAST_OBSERVED_YEAR), "cmhc_fixed_sample_growth"],
    f"SCHL échantillon fixe Ottawa {LAST_OBSERVED_YEAR}": market_signals.loc[(ONTARIO, LAST_OBSERVED_YEAR), "cmhc_fixed_sample_growth"],
    f"IPC logement Canada {LAST_OBSERVED_YEAR}": market_signals.loc[(QUEBEC, LAST_OBSERVED_YEAR), "cpi_shelter_growth"],
})
market_check_2026.round(2).to_frame("%")

**Lecture :** 

**Lecture :**

- La prévision (3,6 %) est **égale à la hausse effective de 2025** et bien sous la hausse contractuelle du marché à Montréal
  (SCHL, 7,2 %) : l'écart correspond aux concessions, comme en 2024-2025 (6b). Elle est proche de l'inflation du logement au Canada
  en 2025 (3,1 %).


- Elle est cohérente avec les signaux contractuels propres à Équinoxe pour le prochain bail : loyer proposé dans `listings`
  environ 2,5 % au-dessus du dernier bail, TAL 2026 de 3,1 % pour les renouvellements québécois.
- **Ce qu'elle suppose :** les concessions arrêtent de grossir et reculent un peu (les logements de 2026 partent d'un rabais moyen
  de 9,0 %, contre 5,8 % pour les baux de 2025). C'est le principal risque de la prévision (section 8).

## 8. Hypothèses et limites

**Hypothèses**
1. Les **rabais reculent en partie** en 2026 : les logements de 2026 partent d'un rabais élevé, et le modèle a appris qu'après un
   gros rabais, le loyer effectif du bail suivant monte davantage. Si les concessions continuent d'augmenter au rythme de 2025,
   la hausse effective sera plus basse (la variante « sans rabais précédent » donne 2,3 %).
2. La **part de renouvellements** de chaque immeuble reste celle de l'historique.
3. Le prochain bail de chaque logement débute le lendemain de la fin du bail en cours (pas de vacance prolongée).
4. The Met est **exempté** de la ligne directrice ontarienne (immeuble occupé après le 15 novembre 2018), à confirmer.

**Limites**
1. **Les chocs d'année ne sont pas prévisibles avec ces données** : 2023 reste la grande erreur (aucune variable connue fin 2022
   ne l'annonçait). Toutes les variantes testées plafonnent autour d'un point d'erreur annuelle.
2. **Peu d'années** : 7 années de cible, et le taux réglementaire, comme toute série publique annuelle, sert en partie à
   reconnaître l'année. On l'a gardé parce qu'il améliore l'erreur par bail et a un sens économique.
3. **The Met** n'a des paires qu'à partir de 2024 : sa prévision s'appuie surtout sur ce que le modèle apprend du Québec.
4. L'extrait ne permet pas de calculer l'occupation, l'inoccupation ni le revenu total du portefeuille : **aucun de ces chiffres
   n'est tiré des données d'Équinoxe**; l'inoccupation utilisée est celle du marché (SCHL).
5. `equinoxe_listings` (loyers proposés) n'existe que pour la fin de 2025 : impossible de le valider au backtest, il sert
   seulement de vérification (7g).

## Conclusion

> **Hausse 2026 de Collection Équinoxe : 3,6 %** (croissance annualisée du loyer effectif à unité constante, médiane de
> tous les baux 2026, six immeubles), **fourchette 2,5 % à 4,6 %**.

## 9. Références

**Données publiques** (fichiers dans `raw_public_data/` et `public_data/`, 2021-2025)
- SCHL, *Enquête sur les logements locatifs*, tableaux de données du Rapport sur le marché locatif 2021-2025 (`rmr-canada-AAAA-fr.xlsx`) :
  inoccupation, rotation, loyer moyen, univers, variation à échantillon fixe. https://www.cmhc-schl.gc.ca
- Statistique Canada, tableau 18-10-0004-01, *Indice des prix à la consommation mensuel*, Canada. https://www150.statcan.gc.ca
- Statistique Canada, tableau 14-10-0460-01 (population active, RMR) et 17-10-0149-01 (composantes de l'accroissement démographique, RMR).
- Tribunal administratif du logement, calcul annuel de l'augmentation de loyer. https://www.tal.gouv.qc.ca
- Gouvernement de l'Ontario, ligne directrice sur l'augmentation des loyers et exemption des logements occupés après le
  15 novembre 2018. https://www.ontario.ca/page/rent-increase-guideline

**Méthode et outils**
- JADCO, *Clés en main* (consignes) et `starter.ipynb`.
- scikit-learn, `HistGradientBoostingRegressor`. https://scikit-learn.org
- pandas, NumPy, matplotlib, joblib.

**Outils d'IA**
- Claude Code (Anthropic) : aide à l'écriture du code et des textes, à l'extraction des séries publiques, aux tests de variables et à la mise en forme
  du notebook. Les choix de méthode, les vérifications et les conclusions ont été revus par l'équipe.

**Notebooks de travail** (non évalués, détail des analyses) : `exploration`, `unite_constante`, `concessions`,
`renouvellements_relocations`, `donnees_publiques`, `modele_v1`.